# Engram: neural and non-neural search

Compares four ways of finding memories on development conversations 0 to 4, then has the model answer with each. Conversations 5 to 9 stay untouched for a final frozen test.

Choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. Keep this tab open. Use benchmark data only.

In [ ]:
MODEL = "gemma3:4b"
TAG = "dev_g4"
COMMIT = "552c3d4"
import subprocess
print("GPU:", subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip() or "NONE. Choose a GPU runtime and run again.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
SAVE = "/content/drive/MyDrive/engram_results"
os.makedirs(SAVE, exist_ok=True)

In [ ]:
%cd /content
!rm -rf Engram
!git clone -q https://github.com/manjunathhari/Engram.git
%cd /content/Engram
!git checkout -q {COMMIT}
!git clone -q --depth 1 https://github.com/snap-research/locomo.git
!pip install -q wordllama scikit-learn sentence-transformers
!python - <<'P'
import os, shutil, wordllama
p = os.path.dirname(wordllama.__file__)
a, b = os.path.join(p, "tokenizers"), os.path.join(p, "tokenizer")
if os.path.isdir(a) and not os.path.exists(b): shutil.copytree(a, b)
print("embedding model package ready")
P

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="4096", OLLAMA_KEEP_ALIVE="-1")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2); break
    except Exception:
        time.sleep(1)
!ollama pull {MODEL}
!ollama run {MODEL} "Reply with the single word ready."

In [ ]:
%%writefile neural_retrieval.py
"""Neural and non-neural search on development conversations 0-4, with Gemma answers.
current: Engram as frozen. hybrid: word and meaning search. learned: a small trained network reorders results,
trained on the other four conversations. crossencoder: a pre-trained neural reranker reorders results."""
import argparse,json,math,os,re,sys,time,zlib
from collections import Counter
import numpy as np
sys.path.insert(0,os.getcwd())
import vps_kit as K
import engram_chat as E
from engram import Config,Engrams
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
p=argparse.ArgumentParser();p.add_argument('--url');p.add_argument('--model');p.add_argument('--tag');p.add_argument('--sample',type=int,default=240);p.add_argument('--budget',type=int,default=800);a=p.parse_args()
words=E.words;CAP=E.engram.CAP;TIME=E.engram.TIMEWORD;ROUTES=E.engram.ROUTES
FEEL=set("happy excited proud love loved sad angry scared afraid worried stress lost died cried amazing great awesome wonderful terrible hate fun thrilled grateful".split())
from wordllama import WordLlama
enc=WordLlama.load()
def emb(ts):
    v=np.array(enc.embed(ts),dtype=np.float32);return v/(np.linalg.norm(v,axis=1,keepdims=True)+1e-9)
try:
    from sentence_transformers import CrossEncoder
    ce=CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2');print('reranker ready',flush=True)
except Exception as e:
    ce=None;print('reranker unavailable, continuing without it:',str(e)[:90],flush=True)
class Mem:
    def __init__(s,c):
        s.c=c;t=c.turns;s.N=len(t);s.txt=[x['speaker']+': '+x['text']+(' [image: '+x['cap']+']' if x['cap'] else '') for x in t]
        s.tok=[E.count_tokens(x)+8 for x in s.txt];s.sess=[x['session'] for x in t];s.w=[words(x) for x in s.txt];s.avg=sum(map(len,s.w))/s.N
        s.p={}
        for i,w in enumerate(s.w):
            for k,v in Counter(w).items():s.p.setdefault(k,[]).append((i,v))
        s.vec=emb(s.txt);s.ent={}
        for i,x in enumerate(t):
            for e in CAP.findall(x['text']):
                if e.lower() not in E.STOP:s.ent.setdefault(E.stem(e.lower()),set()).add(i)
        s.next={i:i+1 for i in range(s.N-1) if s.sess[i]==s.sess[i+1]}
        seen=Counter();s.nov=np.zeros(s.N);s.feel=np.zeros(s.N);s.fact=np.zeros(s.N)
        for i,x in enumerate(t):
            ws=set(s.w[i]);s.nov[i]=sum(seen[w]==0 for w in ws)/max(1,len(ws))
            for w in ws:seen[w]+=1
            s.feel[i]=min(1,sum(y in FEEL for y in re.findall(r"[a-z']+",x['text'].lower()))/2);s.fact[i]=min(1,(len(CAP.findall(x['text']))+len(re.findall(r"\b\d",x['text'])))/3)
        s.seen=seen;rng=np.random.default_rng(3);s.U=4096;s.P=(rng.random((s.U,s.vec.shape[1]))<.1).astype(np.float32)
        X=np.stack([s.code(s.vec[i],set(s.w[i])) for i in range(s.N)]);s.X=X/(np.linalg.norm(X,axis=1,keepdims=True)+1e-9)
        m=Config(**json.load(open('final_config.json')));s.eng=Engrams(cfg=m);eps={}
        for x in t:
            if x['session'] not in eps:eps[x['session']]=s.eng.add_episode(x['date'])
            s.eng.add_turn(x['id'],eps[x['session']],x['speaker'],x['text'],x['cap'])
        s.eng.sleep();s.pos={x['id']:i for i,x in enumerate(t)}
    def code(s,v,ws):
        x=np.zeros(s.U,dtype=np.float32);x[np.argpartition(-(s.P@v),48)[:48]]=1
        for w in ws:
            for j in range(4):x[zlib.crc32(f'{w}|{j}'.encode())%s.U]+=math.log(1+s.N/(1+s.seen[w]))
        return x
    def bm25(s,q):
        r=np.zeros(s.N)
        for w in set(words(q)):
            pl=s.p.get(w)
            if not pl:continue
            idf=math.log(1+(s.N-len(pl)+.5)/(len(pl)+.5))
            for i,tf in pl:r[i]+=idf*tf*2.2/(tf+1.2*(.5+.5*len(s.w[i])/s.avg))
        return r
    def sig(s,q):
        n=len(set(words(q)));return (5.,.7) if n<=3 else (7.,.6) if n<=6 else (9.,.5) if n<=9 else (10.,.5) if n<=15 else (12.,.5)
    def ents(s,q):
        r=np.zeros(s.N)
        for e in CAP.findall(q):
            if e.lower() in E.STOP:continue
            st=s.ent.get(E.stem(e.lower()),())
            if st:
                w=.5/(1+.001*(len(st)-1)**2)
                for i in st:r[i]=max(r[i],w)
        return r
    def feats(s,q,K_=120):
        qv=emb([q])[0];b=s.bm25(q);cos=s.vec@qv;mid,st=s.sig(q);bn=np.where(b>0,1/(1+np.exp(-st*(b-mid))),0);ent=s.ents(q);h=cos+bn;f=h+ent
        x=s.code(qv,set(words(q)));x/=np.linalg.norm(x)+1e-9;na=s.X@x;cand=np.argsort(-f)[:K_];qw=set(words(q));when=bool(ROUTES['time'].search(q));vis=bool(ROUTES['visual'].search(q));names={y.lower() for y in CAP.findall(q)}
        sc={}
        for i in np.argsort(-f)[:40]:sc[s.sess[i]]=sc.get(s.sess[i],0)+max(f[i],0)
        top=max(sc.values()) if sc else 1;F=[]
        for i in cand:
            t=s.c.turns[i];ws=set(s.w[i]);rare=max((math.log(1+s.N/(1+len(s.p[w]))) for w in qw&ws),default=0);prev=s.c.turns[i-1]['text'] if i else ''
            F.append([cos[i],bn[i],b[i],ent[i],na[i],f[i],len(qw&ws)/max(1,len(qw)),rare,len(ws),len(t['text'])/200,s.feel[i],s.nov[i],s.fact[i],float(bool(t['cap'])),float(bool(TIME.search(t['text']))),float(when),float(when and TIME.search(t['text']) is not None),float(vis),float(vis and bool(t['cap'])),float(t['speaker'].lower() in names),float(prev.strip().endswith('?')),float(t['text'].strip().endswith('?')),sc.get(s.sess[i],0)/(top+1e-9),len(qw)/10])
        return cand,np.array(F,dtype=np.float32),h
    def pick(s,o,B):
        out,seen,u=[],set(),0
        for i in o:
            if u>=B:break
            for j in (i,s.next.get(i)):
                if j is None or j in seen:continue
                seen.add(j);out.append(j);u+=s.tok[j]
        return out,u
    def text(s,ids):
        L=[]
        for i in ids:
            t=s.c.turns[i];L.append(f"[{t['date']}] {t['speaker']}: "+E.with_dates(t['text'],t['date'])+(f" [image: {t['cap']}]" if t['cap'] else ''))
        return "\n".join(L)
ids=[0,1,2,3,4];convs={i:K.load_conv(K.DATA[i]) for i in ids};print('building memories',flush=True);M={i:Mem(convs[i]) for i in ids}
D={}
for ci in ids:
    D[ci]={}
    for i,q in enumerate(convs[ci].qs):
        if q['cat']==5 or not q['ev']:continue
        cand,F,h=M[ci].feats(q['q']);ev={M[ci].pos[e] for e in q['ev']};D[ci][i]=(cand,F,h,np.array([1 if x in ev else 0 for x in cand]),ev)
print('questions',sum(len(v) for v in D.values()),flush=True)
NET={}
for ci in ids:
    X=np.vstack([v[1] for cj in ids if cj!=ci for v in D[cj].values()]);Y=np.concatenate([v[3] for cj in ids if cj!=ci for v in D[cj].values()]);pos=np.where(Y==1)[0];neg=np.where(Y==0)[0];rng=np.random.default_rng(1);neg=rng.choice(neg,min(len(neg),len(pos)*8),replace=False);sel=np.concatenate([pos,neg]);sc=StandardScaler().fit(X[sel]);NET[ci]=(sc,MLPClassifier((32,16),max_iter=300,random_state=1,early_stopping=True).fit(sc.transform(X[sel]),Y[sel]))
def order(system,ci,i,q):
    cand,F,h,_,_=D[ci][i];mem=M[ci]
    if system=='hybrid':return list(np.argsort(-h))
    if system=='learned':
        sc,clf=NET[ci];pr=clf.predict_proba(sc.transform(F))[:,1];r=list(cand[np.argsort(-pr)]);have=set(r);return r+[x for x in np.argsort(-h) if x not in have]
    if system=='crossencoder':
        top=list(np.argsort(-h)[:80]);pr=ce.predict([(q['q'],mem.txt[x]) for x in top],batch_size=64,show_progress_bar=False);r=[top[j] for j in np.argsort(-pr)];have=set(r);return r+[x for x in np.argsort(-h) if x not in have]
systems=['current','hybrid','learned']+(['crossencoder'] if ce else [])
def ret(system,ci,i,q):
    mem=M[ci]
    if system=='current':
        ns=mem.eng.retrieve(q['q'],None,use_habit=False);r=[mem.pos[mem.eng.engrams[n].id] for n in ns];return r,sum(mem.tok[x] for x in r)
    return mem.pick(order(system,ci,i,q),a.budget)
os.makedirs('results',exist_ok=True);ev={}
for s in systems:
    rec=[];cat=[];tok=[];t0=time.perf_counter()
    for ci in ids:
        for i,q in enumerate(convs[ci].qs):
            if i not in D[ci]:continue
            r,u=ret(s,ci,i,q);e=D[ci][i][4];rec.append(len(set(r)&e)/len(e));cat.append(q['cat']);tok.append(u)
    rec=np.array(rec);cat=np.array(cat);ev[s]=dict(n=len(rec),evidence=round(rec.mean()*100,1),tokens=round(np.mean(tok)),ms=round((time.perf_counter()-t0)/len(rec)*1000,1),single=round(rec[cat==4].mean()*100,1),multi=round(rec[cat==1].mean()*100,1),temporal=round(rec[cat==2].mean()*100,1),inference=round(rec[cat==3].mean()*100,1));print(s,ev[s],flush=True)
json.dump(ev,open(f'results/neural_{a.tag}_evidence.json','w'),indent=1)
llm=K.LLM(a.url,a.model,'local',1200);path=f'results/neural_{a.tag}.jsonl';done={(r['conv'],r['i'],r['system']) for r in map(json.loads,open(path))} if os.path.exists(path) else set()
plan=K.balanced(convs,ids,a.sample);print(len(plan),'questions to answer with',systems,flush=True);t0=time.time()
with open(path,'a') as out:
    for n,(ci,i) in enumerate(plan,1):
        q=convs[ci].qs[i]
        for s in systems:
            if (ci,i,s) in done:continue
            r,u=ret(s,ci,i,q);e=D[ci][i][4];ctx=M[ci].text(r) or '(no excerpts)';pred,use,secs=llm(K.ANSWER.format(ctx=ctx,q=q['q']),max_tokens=60)
            out.write(json.dumps(dict(conv=ci,i=i,system=s,cat=K.CATS[q['cat']],q=q['q'],gold=q['answer'],pred=pred,recall=len(set(r)&e)/len(e),ctx_tokens=E.count_tokens(ctx),secs=secs))+'\n');out.flush()
        if n%10==0:print(f'question {n}/{len(plan)}',flush=True)
print('Done',flush=True)

In [ ]:
import os, shutil, subprocess, sys
os.chdir("/content/Engram")
os.makedirs("results", exist_ok=True)
LOCAL, DRIVE = f"results/neural_{TAG}.jsonl", f"{SAVE}/neural_{TAG}.jsonl"
EV = f"results/neural_{TAG}_evidence.json"
if os.path.exists(DRIVE): shutil.copy(DRIVE, LOCAL)
run = subprocess.Popen([sys.executable, "-u", "neural_retrieval.py", "--url", "http://localhost:11434/v1", "--model", MODEL, "--tag", TAG], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in run.stdout:
    print(line, end="")
    if line.startswith("question ") and os.path.exists(LOCAL): shutil.copy(LOCAL, DRIVE)
code = run.wait()
for f in (LOCAL, EV):
    if os.path.exists(f): shutil.copy(f, SAVE)
print("Finished" if code == 0 else f"Stopped with an error, code {code}. Run this cell again to continue.")

When it says Finished, send Claude `neural_dev_g4.jsonl` and `neural_dev_g4_evidence.json` from the `engram_results` Drive folder.